In [39]:
import hashlib
import json
import subprocess
import time
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split


In [40]:
# 1. Pipeline Configuration
WORKSPACE = Path('./')
DATA_PATH = WORKSPACE / 'data/manual_labels_with_hal_metadata.csv'
ARTIFACT_DIR = WORKSPACE / 'mpnet_sgd_artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
CLASSIFIER_PATH = ARTIFACT_DIR / 'sgd_classifier.joblib'
ENCODER_PATH = ARTIFACT_DIR / 'all-mpnet-base-v2'
SUBAXIS_PATH = ARTIFACT_DIR / 'subaxis_references.joblib'
METADATA_PATH = ARTIFACT_DIR / 'metadata.json'

RANDOM_STATE = 42


SEQUOIA_AXES = {
    "Core AI": "core AI foundations learning secure trustworthy frugal embedded hybrid explainable AI hardware signal image language robotics theory human AI interaction regulation ethics",
    "AI, Cybersecurity and Defense": "AI cybersecurity defense systems infrastructures information intelligence digital influence security adversarial attacks privacy anomaly detection networks cyber defense disinformation robotics",
    "AI, Environment and Ocean": "AI environment ocean earth observation modeling climate remote sensing digital twins data assimilation uncertainty marine pollution maritime traffic ocean dynamics",
}

SEQUOIA_SAXES = {

    "Core AI": {

        "Theoretical and Conceptual Foundations of AI": [
            "generalization",
            "statistical bounds",
            "continual learning",
            "optimal transport",
            "long-tail distributions",
            "rare events",
            "time series",
            "graphs",
            "geometric deep learning",
        ],

        "Symbolic / Statistical Hybridization": [
            "hybrid AI",
            "neuro-symbolic AI",
            "knowledge graphs",
            "counterfactual explanations",
            "epistemic planning",
            "reasoning",
        ],

        "Formal Evaluation of ML Technologies": [
            "formal methods",
            "model checking",
            "certification",
            "certified robustness",
            "neural network verification",
            "static analysis",
        ],

        "Frugal AI, Edge AI and Hardware Architectures": [
            "frugal AI",
            "quantization",
            "sparsity",
            "AI accelerators",
            "model compression",
            "energy efficiency",
            "edge AI",
            "compilation",
        ],

        "Human–AI Interaction and Explainability": [
            "XAI",
            "explainability",
            "trust",
            "confidence measures",
            "human-AI interaction",
            "attention mechanisms",
            "SHAP",
            "interactive learning",
        ],

        "Acceptability, Responsibility, Regulation and Ethics": [
            "AI Act",
            "AI regulation",
            "responsibility",
            "bias",
            "AI ethics",
            "law",
            "organizations",
            "AI use",
            "acceptability",
        ],

        "Signal, Image and Language": [
            "signal processing",
            "computer vision",
            "natural language processing",
            "NLP",
            "large language models",
            "LLMs",
            "diffusion models",
            "multimodal AI",
            "vision-language models",
            "image compression",
        ],
    },


    "AI, Cybersecurity and Defense": {

        "AI for Cybersecurity": [
            "intrusion detection",
            "anomaly detection",
            "vulnerabilities",
            "obfuscation",
            "cyber defense",
            "threat intelligence",
            "logs",
            "security monitoring",
        ],

        "AI Security": [
            "adversarial attacks",
            "data poisoning",
            "model inversion",
            "model privacy",
            "model confidentiality",
            "hardware attacks",
            "side-channel attacks",
            "federated learning",
        ],

        "Intelligence and Information Security": [
            "information extraction",
            "disinformation",
            "deepfakes",
            "information manipulation",
            "OSINT",
            "open-source intelligence",
            "fake news",
            "digital influence",
        ],

        "Intelligent and Secure Networks": [
            "computer networks",
            "5G",
            "software-defined networking",
            "SDN",
            "network functions virtualization",
            "NFV",
            "graph neural networks for networks",
            "network orchestration",
            "QoS",
            "quality of service",
        ],

        "Autonomous Robotics and Interaction": [
            "robotics",
            "shared control",
            "robot perception",
            "planning",
            "robot swarms",
            "human-robot interaction",
            "drones",
            "UAVs",
            "underwater robots",
            "autonomous robots",
        ],
    },


    "AI, Environment and Ocean": {

        "Physics-Informed AI": [
            "physics-informed AI",
            "physics-informed neural networks",
            "PINN",
            "data assimilation",
            "conformal prediction",
            "uncertainty quantification",
            "surrogate models",
        ],

        "Observation, Remote Sensing and Data Integration": [
            "remote sensing",
            "SAR",
            "synthetic aperture radar",
            "hyperspectral imaging",
            "sonar",
            "sensors",
            "sensor fusion",
            "data fusion",
            "AIS",
            "weak signals",
            "Earth observation",
        ],

        "Ocean Digital Twin": [
            "digital twin",
            "ocean digital twin",
            "ocean dynamics",
            "oceanography",
            "ocean modeling",
            "ocean-atmosphere modeling",
            "marine extremes",
            "underwater mapping",
            "subsea mapping",
        ],
    },
}



# Set tracking URI and experiment[cite: 1]
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('sequoia-mpnet-sgd-continual')

<Experiment: artifact_location='/home/formation2026/Bureau/codeworkspace/veille_test/mlruns/2', creation_time=1790689867267, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1790689867267, lifecycle_stage='active', name='sequoia-mpnet-sgd-continual', tags={}, trace_location=None, workspace='default'>

In [41]:
# 2. DVC Data Synchronization
print("🔄 Checking & pulling latest dataset version via DVC...")
try:
    subprocess.run(["dvc", "pull", str(DATA_PATH)], check=True, cwd=str(WORKSPACE))
    print("✅ DVC dataset pull completed successfully.")
except Exception as e:
    print(f"⚠️ DVC pull failed or skipped: {e}")

🔄 Checking & pulling latest dataset version via DVC...
⚠️ DVC pull failed or skipped: [Errno 2] No such file or directory: 'dvc'


In [ ]:
# 3. Load & Prepare Updated Dataset
df = pd.read_csv(DATA_PATH, dtype={'halId_s': str})

text_columns = ['title_s', 'abstract', 'keywords']
df['text_for_classification'] = (
    df[text_columns]
    .fillna('')
    .astype(str)
    .agg(' '.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)



# Split holdout test set to evaluate candidate vs model performance[cite: 1]
X_train_df, X_test_df = train_test_split(
    df,
    test_size=0.25,
    stratify=df['manual_label'],
    random_state=RANDOM_STATE
)

print(f"Training set records: {len(X_train_df):,}")
print(X_train_df['manual_label'].value_counts())
print("*****************************************")
print(f"Test set records: {len(X_test_df):,}")
print(X_test_df['manual_label'].value_counts())


Training set records: 1,053
manual_label
No class                         465
Core AI                          395
AI, environment and ocean        113
AI, cybersecurity and defense     80
Name: count, dtype: int64
*****************************************
Test set records: 351
manual_label
No class                         155
Core AI                          131
AI, environment and ocean         38
AI, cybersecurity and defense     27
Name: count, dtype: int64


In [ ]:
# ==========================================
# 4. Encoder & Embeddings
# ==========================================
if ENCODER_PATH.exists():
    encoder = SentenceTransformer(ENCODER_PATH,local_files_only=True)
else:
    encoder = SentenceTransformer(MODEL_NAME)
    encoder.save(ENCODER_PATH)

print("⚡ Encoding text samples...")
embedding_started = time.perf_counter()

train_embeddings = encoder.encode(
    X_train_df['text_for_classification'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
test_embeddings = encoder.encode(
    X_test_df['text_for_classification'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

embedding_seconds = time.perf_counter() - embedding_started


# ============================================================
# 1. Cache MPNet embeddings
# ===========================================================
embedding_texts = X_train_df["text_for_classification"].tolist()
embedding_cache_path = ARTIFACT_DIR / "training_embeddings.npz"
embedding_fingerprint = hashlib.sha256(
    json.dumps(
        {
            "model_name": MODEL_NAME,
            "normalize_embeddings": True,
            "texts": embedding_texts,
        },
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

⚡ Encoding text samples...


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# 2. Start MLflow run
# ============================================================

with mlflow.start_run(run_name="mpnet-sgd-training"):
    # --------------------------------------------------------
    # Load or compute MPNet embeddings
    # --------------------------------------------------------
    embedding_started = time.perf_counter()
    cache_hit = False
    if embedding_cache_path.exists():
        with np.load(
            embedding_cache_path,
            allow_pickle=False
        ) as cache:
            if str(cache["fingerprint"].item()) == embedding_fingerprint:
                embeddings = cache["embeddings"]
                cache_hit = True
    if cache_hit:
        print(f"Loaded cached embeddings from {embedding_cache_path}")
    else:
        embeddings = encoder.encode(
            embedding_texts,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        np.savez_compressed(
            embedding_cache_path,
            embeddings=embeddings,
            fingerprint=np.array(embedding_fingerprint),
        )
        print(f"Saved encoded embeddings to {embedding_cache_path}")
    embedding_seconds = time.perf_counter() - embedding_started

    # ==========================================
    # 5. Continual Learning Update (partial_fit)
    # ==========================================
    print("\n🧠 Updating SGDClassifier via incremental learning...")
    if CLASSIFIER_PATH.exists():
        classifier = joblib.load(CLASSIFIER_PATH)
        print("✅ Existing model weights loaded.")
    else:
        classifier = SGDClassifier(
            loss='log_loss',
            max_iter=2000,
            tol=1e-2,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        )
        print("ℹ️ New SGDClassifier initialized.")

    fit_started = time.perf_counter()

    # Perform online incremental update
    classifier.partial_fit(
        train_embeddings,
        X_train_df['manual_label'],
        classes=np.array([
    "AI, cybersecurity and defense",
    "AI, environment and ocean",
    "Core AI",
    "No class"
    ])
                        )

    fit_seconds = time.perf_counter() - fit_started

    # Evaluate candidate model on holdout set
    test_preds = classifier.predict(test_embeddings)
    test_acc = accuracy_score(X_test_df['manual_label'], test_preds)
    test_macro_f1 = f1_score(X_test_df['manual_label'], test_preds, average='macro')

    print(f"\n================ HOLDOUT EVALUATION ================")
    print(classification_report(X_test_df['manual_label'], test_preds, zero_division=0))

Loaded cached embeddings from mpnet_sgd_artifacts/training_embeddings.npz

🧠 Updating SGDClassifier via incremental learning...
✅ Existing model weights loaded.

================ HOLDOUT EVALUATION ================
                               precision    recall  f1-score   support

AI, cybersecurity and defense       0.83      0.93      0.88        27
    AI, environment and ocean       0.80      0.87      0.84        38
                      Core AI       0.90      0.89      0.89       131
                     No class       0.91      0.88      0.90       155

                     accuracy                           0.89       351
                    macro avg       0.86      0.89      0.88       351
                 weighted avg       0.89      0.89      0.89       351



In [58]:
# 6. Re-building Pillar Sub-axis 
# ==========================================
print("🔄 Updating pillar sub-axis embeddings...")
subaxis_references = {}
for pillar, subaxes in SEQUOIA_SAXES.items():
    pillar_description = SEQUOIA_AXES[pillar]
    subaxis_names = list(subaxes)
    subaxis_texts = [
        f"{' '.join(keywords)} {pillar_description}"
        for keywords in subaxes.values()
    ]

    subaxis_references[pillar] = {
        'names': subaxis_names,
        'embeddings': encoder.encode(
            subaxis_texts,
            normalize_embeddings=True,
            show_progress_bar=False,
        ),
    }

🔄 Updating pillar sub-axis embeddings...


In [ ]:
# ==========================================
# 7. Log Run & Export Deployable Artifacts
# ==========================================
with mlflow.start_run(run_name="continual-learning-dvc-update"):
    # Log hyperparameters and metrics to MLflow[cite: 1]
    mlflow.log_params({
        "dataset_rows": len(df),
        "learning_mode": "incremental_partial_fit",
        "loss": "log_loss",
        "dvc_tracked": True,
        "random_state": RANDOM_STATE,
    })

    mlflow.log_metrics({
        "holdout_accuracy": test_acc,
        "holdout_macro_f1": test_macro_f1,
        "embedding_time_sec": embedding_seconds,
        "fit_time_sec": fit_seconds,
    })

    # Save artifacts locally
    joblib.dump(classifier, CLASSIFIER_PATH)
    encoder.save(str(ENCODER_PATH))
    joblib.dump(subaxis_references, SUBAXIS_PATH)

    metadata = {
        'model_name': MODEL_NAME,
        'classifier': 'sklearn.linear_model.SGDClassifier',
        'loss': 'log_loss',
        'labels': CANONICAL_LABELS,
        'text_columns': text_columns,
        'subaxis_source': 'hal_sequoia_extractor.SEQUOIA_SAXES',
        'random_state': RANDOM_STATE,
        'last_updated_samples': len(df),
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

    mlflow.sklearn.log_model(classifier, artifact_path="model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026/09/29 16:59:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [ ]:


import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)




WORKSPACE = Path('/home/formation2026/Bureau/codeworkspace/veille_test/')
DATA_PATH = WORKSPACE / 'data/manual_labels_with_hal_metadata.csv'
ARTIFACT_DIR = WORKSPACE / 'mpnet_sgd_artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
# Keep tracking data on this machine beside the deployment artifacts.
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('sequoia-mpnet-sgd-local')
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2' 
RANDOM_STATE = 42
CANONICAL_LABELS = [
    'Core AI',
    'AI, Cybersecurity and Defense',
    'AI, Environment and Ocean',
    'No class',
]
PILLAR_MAP = {
    'Core AI': 'Core AI',
    'IA & cybersécurité': 'AI, Cybersecurity and Defense',
    'IA & environnement': 'AI, Environment and Ocean',
    'Non classifié': 'No class',
}
print(DATA_PATH)
print(ARTIFACT_DIR)

/home/formation2026/Bureau/codeworkspace/veille_test/data/manual_labels_with_hal_metadata.csv
/home/formation2026/Bureau/codeworkspace/veille_test/mpnet_sgd_artifacts


## 2. Load and prepare the curated training data

In [14]:
df = pd.read_csv(DATA_PATH, dtype={'halId_s': str})


text_columns = ['title_s', 'abstract', 'keywords']
df['text_for_classification'] = df[text_columns].fillna('').astype(str).agg(' '.join, axis=1).str.replace(r'\s+', ' ', regex=True).str.strip()

training_df = df.copy()
print(f'Total records: {len(df):,}')
print(f'Training records: {len(training_df):,}')
print(training_df['manual_label'].value_counts())

Total records: 1,404
Training records: 1,404
manual_label
No class                         620
Core AI                          526
AI, environment and ocean        151
AI, cybersecurity and defense    107
Name: count, dtype: int64


In [15]:
df.head(1)


,halId_s,title_s,manual_label,abstract,keywords,text_for_classification
0,hal-04164424,Unravelling individual rhythmic abilities usin...,Core AI,Humans can easily extract the rhythm of a comp...,Human behaviour | Psychology,Unravelling individual rhythmic abilities usin...


## Local MLflow profiling

Training runs and timing metrics are recorded locally under `mpnet_sgd_artifacts/mlruns`. Open the run details with `mlflow ui --backend-store-uri mpnet_sgd_artifacts/mlruns` from the workspace directory. No hosted tracking server is used.


## 3. Encode with MPNet and train SGDClassifier

In [ ]:
encoder = SentenceTransformer(MODEL_NAME)
encoder.save("./all-mpnet-base-v2")
print("Model saved locally.")

encoder = SentenceTransformer(
    "./all-mpnet-base-v2",
    local_files_only=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved locally.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [24]:
MAX_Iter = 2000
tol = 1e-2

In [ ]:
# ============================================================
# 2. Start MLflow run
# ============================================================

with mlflow.start_run(run_name="mpnet-sgd-training"):
    # --------------------------------------------------------
    # Load or compute MPNet embeddings
    # --------------------------------------------------------
    embedding_started = time.perf_counter()
    cache_hit = False
    if embedding_cache_path.exists():
        with np.load(
            embedding_cache_path,
            allow_pickle=False
        ) as cache:
            if str(cache["fingerprint"].item()) == embedding_fingerprint:
                embeddings = cache["embeddings"]
                cache_hit = True
    if cache_hit:
        print(f"Loaded cached embeddings from {embedding_cache_path}")
    else:
        embeddings = encoder.encode(
            embedding_texts,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        np.savez_compressed(
            embedding_cache_path,
            embeddings=embeddings,
            fingerprint=np.array(embedding_fingerprint),
        )
        print(f"Saved encoded embeddings to {embedding_cache_path}")
    embedding_seconds = time.perf_counter() - embedding_started
    # ========================================================
    # 3. Train/test split
    # ========================================================
    X_train, X_test, y_train, y_test = train_test_split(
        embeddings,
        training_df["manual_label"],
        test_size=0.25,
        stratify=training_df["manual_label"],
        random_state=RANDOM_STATE,
    )
    # ========================================================
    # 4. SGD classifier
    # ========================================================
    classifier = SGDClassifier(
        loss="log_loss",
        max_iter=MAX_Iter,
        tol=tol,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    # ========================================================
    # 5. Train classifier
    # ========================================================
    fit_started = time.perf_counter()
    classifier.fit(
        X_train,
        y_train
    )
    fit_seconds = time.perf_counter() - fit_started
    # ========================================================
    # 6. Test-set evaluation
    # ========================================================
    test_prediction = classifier.predict(X_test)
    test_accuracy = accuracy_score(
        y_test,
        test_prediction
    )
    macro_f1 = f1_score(
        y_test,
        test_prediction,
        average="macro"
    )
    weighted_f1 = f1_score(
        y_test,
        test_prediction,
        average="weighted"
    )
    macro_precision = precision_score(
        y_test,
        test_prediction,
        average="macro",
        zero_division=0
    )
    macro_recall = recall_score(
        y_test,
        test_prediction,
        average="macro",
        zero_division=0
    )
    # ========================================================
    # 7. Detailed classification report
    # ========================================================
    report_dict = classification_report(
        y_test,
        test_prediction,
        output_dict=True,
        zero_division=0,
    )
    report_text = classification_report(
        y_test,
        test_prediction,
        zero_division=0,
        digits=4,
    )
    print("\n================ TEST SET ================\n")
    print(report_text)
    # ========================================================
    # 8. Log per-class metrics to MLflow
    # ========================================================
    class_names = sorted(training_df["manual_label"].unique())
    for class_name in class_names:
        # MLflow metric names cannot contain problematic characters
        metric_name = (
            class_name
            .lower()
            .replace(" ", "_")
            .replace(",", "")
            .replace("&", "and")
            .replace("-", "_")
            .replace("/", "_")
        )
        mlflow.log_metrics({
            f"{metric_name}_precision":
                report_dict[class_name]["precision"],

            f"{metric_name}_recall":
                report_dict[class_name]["recall"],

            f"{metric_name}_f1":
                report_dict[class_name]["f1-score"],

            f"{metric_name}_support":
                report_dict[class_name]["support"],
        })
    # ========================================================
    # 9. Confusion matrix
    # ========================================================
    cm = confusion_matrix(
        y_test,
        test_prediction,
        labels=class_names,
    )
    fig, ax = plt.subplots(figsize=(9, 7))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    ).plot(
        ax=ax,
        xticks_rotation=45,
        values_format="d",
    )

    plt.tight_layout()

    confusion_matrix_path = (
        ARTIFACT_DIR / "confusion_matrix.png"
    )

    plt.savefig(
        confusion_matrix_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close()

    mlflow.log_artifact(
        str(confusion_matrix_path),
        artifact_path="evaluation"
    )

    # ========================================================
    # 10. Stratified 5-fold cross-validation
    # ========================================================

    print("\n================ 5-FOLD CV ================\n")

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    cv_results = cross_validate(
        classifier,
        embeddings,
        training_df["manual_label"],
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "macro_f1": "f1_macro",
            "weighted_f1": "f1_weighted",
            "macro_precision": "precision_macro",
            "macro_recall": "recall_macro",
        },
        n_jobs=-1,
        return_train_score=False,
    )

    cv_accuracy_mean = cv_results["test_accuracy"].mean()
    cv_accuracy_std = cv_results["test_accuracy"].std()

    cv_macro_f1_mean = cv_results["test_macro_f1"].mean()
    cv_macro_f1_std = cv_results["test_macro_f1"].std()

    cv_weighted_f1_mean = cv_results["test_weighted_f1"].mean()
    cv_weighted_f1_std = cv_results["test_weighted_f1"].std()

    cv_macro_precision_mean = (
        cv_results["test_macro_precision"].mean()
    )

    cv_macro_recall_mean = (
        cv_results["test_macro_recall"].mean()
    )
    # Print individual folds

    print(
        "CV Accuracy:",
        np.round(cv_results["test_accuracy"], 4)
    )

    print(
        "CV Macro-F1:",
        np.round(cv_results["test_macro_f1"], 4)
    )

    print(
        "CV Weighted-F1:",
        np.round(cv_results["test_weighted_f1"], 4)
    )

    print(
        f"\nMean CV Macro-F1: "
        f"{cv_macro_f1_mean:.4f} "
        f"+/- {cv_macro_f1_std:.4f}"
    )


    # ========================================================
    # 11. Save CV results
    # ========================================================

    cv_report_path = ARTIFACT_DIR / "cv_results.txt"

    cv_report = f"""
5-Fold Stratified Cross Validation
==================================

Accuracy:
  Fold scores: {cv_results["test_accuracy"]}
  Mean: {cv_accuracy_mean:.4f}
  Std:  {cv_accuracy_std:.4f}

Macro-F1:
  Fold scores: {cv_results["test_macro_f1"]}
  Mean: {cv_macro_f1_mean:.4f}
  Std:  {cv_macro_f1_std:.4f}

Weighted-F1:
  Fold scores: {cv_results["test_weighted_f1"]}
  Mean: {cv_weighted_f1_mean:.4f}
  Std:  {cv_weighted_f1_std:.4f}

Macro Precision:
  Mean: {cv_macro_precision_mean:.4f}

Macro Recall:
  Mean: {cv_macro_recall_mean:.4f}
"""

    cv_report_path.write_text(
        cv_report,
        encoding="utf-8"
    )

    mlflow.log_artifact(
        str(cv_report_path),
        artifact_path="evaluation"
    )


    # ========================================================
    # 12. MLflow parameters
    # ========================================================

    mlflow.log_params({

        "model_name": MODEL_NAME,

        "classifier": "SGDClassifier",

        "loss": "log_loss",

        "max_iter": MAX_Iter,

        "tol": tol,

        "class_weight": "balanced",

        "test_size": 0.25,

        "random_state": RANDOM_STATE,

        "training_rows": len(training_df),

        "embedding_dimensions": embeddings.shape[1],

        "cv_folds": 5,

        "normalize_embeddings": True,

    })


    # ========================================================
    # 13. MLflow metrics
    # ========================================================

    mlflow.log_metrics({
        # Performance
        "test_accuracy": test_accuracy,

        "test_macro_f1": macro_f1,

        "test_weighted_f1": weighted_f1,

        "test_macro_precision": macro_precision,

        "test_macro_recall": macro_recall,

        # CV
        "cv_accuracy_mean": cv_accuracy_mean,

        "cv_accuracy_std": cv_accuracy_std,

        "cv_macro_f1_mean": cv_macro_f1_mean,

        "cv_macro_f1_std": cv_macro_f1_std,

        "cv_weighted_f1_mean": cv_weighted_f1_mean,

        "cv_weighted_f1_std": cv_weighted_f1_std,

        "cv_macro_precision_mean": cv_macro_precision_mean,

        "cv_macro_recall_mean": cv_macro_recall_mean,

        # Runtime
        "embedding_seconds": embedding_seconds,

        "embedding_cache_hit": int(cache_hit),

        "fit_seconds": fit_seconds,

        "total_profiled_seconds":
            embedding_seconds + fit_seconds,

        # Dataset
        "train_rows": len(y_train),

        "test_rows": len(y_test),

    })


    # ========================================================
    # 14. Save classification report
    # ========================================================

    report_path = (
        ARTIFACT_DIR / "classification_report.txt"
    )

    report_path.write_text(
        report_text,
        encoding="utf-8"
    )

    mlflow.log_artifact(
        str(report_path),
        artifact_path="evaluation"
    )


    # ========================================================
    # 15. Save model
    # ========================================================

    mlflow.sklearn.log_model(
        classifier,
        artifact_path="model"
    )


    # ========================================================
    # 16. Final console output
    # ========================================================

    print("\n================ FINAL ================\n")

    print(
        f"Embedding time: "
        f"{embedding_seconds:.2f}s"
    )

    print(
        f"Classifier fit time: "
        f"{fit_seconds:.2f}s"
    )

    print(
        f"Test Accuracy: "
        f"{test_accuracy:.3f}"
    )

    print(
        f"Test Macro-F1: "
        f"{macro_f1:.3f}"
    )

    print(
        f"Test Weighted-F1: "
        f"{weighted_f1:.3f}"
    )

    print(
        f"CV Macro-F1: "
        f"{cv_macro_f1_mean:.3f} "
        f"+/- {cv_macro_f1_std:.3f}"
    )

    print(
        f"CV Accuracy: "
        f"{cv_accuracy_mean:.3f} "
        f"+/- {cv_accuracy_std:.3f}"
    )

Loaded cached embeddings from /home/formation2026/Bureau/codeworkspace/veille_test/mpnet_sgd_artifacts/training_embeddings.npz

================ TEST SET ================

                               precision    recall  f1-score   support

AI, cybersecurity and defense     0.5758    0.7037    0.6333        27
    AI, environment and ocean     0.5439    0.8158    0.6526        38
                      Core AI     0.9140    0.6489    0.7589       131
                     No class     0.7857    0.8516    0.8173       155

                     accuracy                         0.7607       351
                    macro avg     0.7048    0.7550    0.7156       351
                 weighted avg     0.7913    0.7607    0.7636       351


================ 5-FOLD CV ================



2026/09/29 12:10:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


CV Accuracy: [0.7936 0.8043 0.7865 0.79   0.7857]
CV Macro-F1: [0.7706 0.7886 0.7569 0.7527 0.7368]
CV Weighted-F1: [0.7936 0.8046 0.7867 0.7899 0.7876]

Mean CV Macro-F1: 0.7611 +/- 0.0175

================ FINAL ================

Embedding time: 0.02s
Classifier fit time: 0.15s
Test Accuracy: 0.761
Test Macro-F1: 0.716
Test Weighted-F1: 0.764
CV Macro-F1: 0.761 +/- 0.017
CV Accuracy: 0.792 +/- 0.007


## 4. Refit on all labels and save deployment artifacts

In [ ]:
final_classifier = SGDClassifier(
    loss='log_loss',
    max_iter=MAX_Iter,
    tol=tol,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)  
final_classifier.fit(embeddings, training_df['manual_label'])

joblib.dump(final_classifier, ARTIFACT_DIR / 'sgd_classifier.joblib')
encoder.save(str(ARTIFACT_DIR / 'mpnet_encoder'))
(ARTIFACT_DIR / 'metadata.json').write_text(
    json.dumps({
        'model_name': MODEL_NAME,
        'classifier': 'sklearn.linear_model.SGDClassifier',
        'loss': 'log_loss',
        'labels': CANONICAL_LABELS,
        'text_columns': text_columns,
        'subaxis_source': 'hal_sequoia_extractor.SEQUOIA_SAXES',
        'random_state': RANDOM_STATE,
    }, indent=2),
    encoding='utf-8',
)
print('Saved:', ARTIFACT_DIR / 'mpnet_encoder')
print('Saved:', ARTIFACT_DIR / 'sgd_classifier.joblib')
print('Saved:', ARTIFACT_DIR / 'metadata.json')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts/mpnet_encoder
Saved: /home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts/sgd_classifier.joblib
Saved: /home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts/metadata.json


## 5. Build and save pillar-conditioned sub-axis references

For each predicted pillar, only its `SEQUOIA_SAXES` entries are candidates. Both the article and
the sub-axis reference include the corresponding pillar description before MPNet cosine similarity.

In [ ]:
subaxis_references = {}
for pillar, subaxes in SEQUOIA_SAXES.items():
    pillar_description = SEQUOIA_AXES[pillar]
    subaxis_names = list(subaxes)
    subaxis_texts = [
        f"{' '.join(keywords)} {pillar_description}"
        for keywords in subaxes.values()
    ]

    subaxis_references[pillar] = {
        'names': subaxis_names,
        'embeddings': encoder.encode(
            subaxis_texts,
            normalize_embeddings=True,
            show_progress_bar=False,
        ),
    }

joblib.dump(subaxis_references, ARTIFACT_DIR / 'subaxis_references.joblib')
print('Saved:', ARTIFACT_DIR / 'subaxis_references.joblib')
print({pillar: len(reference['names']) for pillar, reference in subaxis_references.items()})

Saved: /home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts/subaxis_references.joblib
{'Core AI': 7, 'AI, Cybersecurity and Defense': 5, 'AI, Environment and Ocean': 3}


## 5. Server-ready prediction function

In [23]:
def predict_article(title='', abstract='', keywords='', journal=''):
    text = ' '.join(str(value or '') for value in [title, abstract, keywords, journal]).strip()
    article_embedding = encoder.encode([text], normalize_embeddings=True)

    pillar_probabilities = final_classifier.predict_proba(article_embedding)[0]
    pillar_index = int(pillar_probabilities.argmax())
    pillar = str(final_classifier.classes_[pillar_index])

    print(f'Predicted pillar: {pillar} (confidence: {pillar_probabilities[pillar_index]:.3f})')
    if pillar not in subaxis_references:
        print(f'Warning: No subaxis references found for pillar "{pillar}". Returning only pillar prediction.')
        return {
            'pillar': pillar,
            'pillar_confidence': float(pillar_probabilities[pillar_index]),
            'subaxis': 'No class',
            'subaxis_similarity': None,
            'pillar_probabilities': {str(label): float(score) for label, score in zip(final_classifier.classes_, pillar_probabilities)},
        }

    #pillar_description = SEQUOIA_AXES[pillar]
    #element_with_pillar = f'{text} {pillar_description}'
    element_without_pillar = text
    element_embedding = encoder.encode([element_without_pillar], normalize_embeddings=True)
    reference = subaxis_references[pillar]
    subaxis_scores = encoder.similarity(element_embedding, reference['embeddings']).cpu().numpy()[0]
    subaxis_index = int(subaxis_scores.argmax())
    subaxis_probabilities = np.exp(subaxis_scores) / np.sum(np.exp(subaxis_scores))  # Softmax to get probabilities
    return {
        'pillar': pillar,
        'pillar_confidence': float(pillar_probabilities[pillar_index]),
        'subaxis': reference['names'][subaxis_index],
        'subaxis_similarity': float(subaxis_scores[subaxis_index]),
        'pillar_probabilities': {str(label): float(score) for label, score in zip(final_classifier.classes_, pillar_probabilities)},
        'subaxis_probabilities': {str(label): float(score) for label, score in zip(reference['names'], subaxis_probabilities)}
        }

predict_article(
    title='AI-based intrusion detection for secure networks',
    abstract='We study anomaly detection and cyber defense using machine learning.',
    keywords='cybersecurity, intrusion detection, anomaly detection',
)

Predicted pillar: AI, cybersecurity and defense (confidence: 0.936)


{'pillar': 'AI, cybersecurity and defense',
 'pillar_confidence': 0.9359273910522461,
 'subaxis': 'No class',
 'subaxis_similarity': None,
 'pillar_probabilities': {'AI, cybersecurity and defense': 0.9359273910522461,
  'AI, environment and ocean': 0.014157047495245934,
  'Core AI': 0.011076272465288639,
  'No class': 0.03883932903409004}}